# GINN v2: more data, smaller network, regularisation, VIX
Mkt-RF daily, 1990 to Aug 2026. Splits by date, never shuffled:
train 1990-2014 (fit everything), validation 2015-2019 (early stopping and settings), test 2020-2026 (touched once).

Baselines fitted on train only, then frozen: GJR and GJR-X (GJR plus δ·VIX²). Two networks on top of GJR:
GINN (inputs r, |r|, log h_GJR) and GINN-VIX (same plus log VIX²).
Questions: does GINN beat GJR? Does GINN-VIX beat GJR-X, i.e. add something beyond VIX itself?

v1 (2,500 days) is unchanged in `GINN.ipynb`. Expected run time: GJR fits a few minutes each, the network grid 10-20 minutes on CPU.

In [1]:
import copy
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

nnf = torch.nn.functional

## 1. Data: returns and VIX, aligned by date
VIX is the close of each day; a feature for day t only ever uses days up to t-1. The few market days with no VIX are forward-filled.

In [2]:
d = pd.read_parquet('../../data/ff_daily_garchdata_long.parquet')
d = d[d['series'] == 'Mkt-RF'].copy()
date_col = 'Date' if 'Date' in d.columns else 'date'      # the date column, whatever its exact name
try:
    d['date'] = pd.to_datetime(d[date_col], format='%d-%m-%Y')   # stored as text like 01-07-1963
except (ValueError, TypeError):
    d['date'] = pd.to_datetime(d[date_col])                       # already a date, or ISO text

vix = pd.read_csv('../../data/VIXCLS.csv')
vix.columns = ['date', 'vix']
vix['date'] = pd.to_datetime(vix['date'])
vix['vix'] = pd.to_numeric(vix['vix'], errors='coerce')

m = d[['date', 'ret']].merge(vix, on='date', how='left')
m = m[m['date'] >= '1990-01-02'].sort_values('date').reset_index(drop=True)
print('market days without a VIX close (forward-filled):', int(m['vix'].isna().sum()))
m['vix'] = m['vix'].ffill()

dates = m['date'].to_numpy()
r_all = torch.tensor(m['ret'].to_numpy() * 100, dtype=torch.float64)          # percent
x_all = torch.tensor(m['vix'].to_numpy() ** 2 / 252, dtype=torch.float64)     # VIX^2 as a daily variance, %^2

TRAIN_END, VAL_END = np.datetime64('2015-01-01'), np.datetime64('2020-01-01')
N_TRAIN = int((dates < TRAIN_END).sum())
r_train, x_train = r_all[:N_TRAIN], x_all[:N_TRAIN]
print(f"{str(dates[0])[:10]} to {str(dates[-1])[:10]} | {len(r_all)} days | train days {N_TRAIN} (expected 9233 and 6301)")

market days without a VIX close (forward-filled): 3
1990-01-02 to 2026-08-31 | 9233 days | train days 6301 (expected 9233 and 6301)


## 2. GJR and GJR-X in torch (one function for both)
h(t) = ω + (α + γ·1{r(t−1)<0})·r²(t−1) + β·h(t−1) + δ·VIX²(t−1).   GJR is the special case δ = 0.
raw has 4 numbers for GJR, 5 for GJR-X. δ = softplus(raw[4])·0.1 ≥ 0. Softmax over [α, γ/2, β, leftover] as before.

In [3]:
def unpack_gjr(raw):
    omega = nnf.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.cat([raw[1:4], torch.zeros(1, dtype=raw.dtype)]), dim=0)   # [alpha, gamma/2, beta, leftover]
    delta = nnf.softplus(raw[4]) * 0.1 if len(raw) > 4 else torch.zeros((), dtype=raw.dtype)
    return omega, w[0], 2 * w[1], w[2], delta


def nll_gjr(raw, r, x):
    omega, alpha, gamma, beta, delta = unpack_gjr(raw)
    h = r.var()
    total = torch.zeros((), dtype=r.dtype)
    for t in range(1, len(r)):
        neg = (r[t - 1] < 0).to(r.dtype)
        h = omega + (alpha + gamma * neg) * r[t - 1] ** 2 + beta * h + delta * x[t - 1]
        total = total + torch.log(h) + r[t] ** 2 / h
    return 0.5 * total / (len(r) - 1)


def fit(raw0, r, x, steps=300, lr=0.05, every=50):
    raw = raw0.clone().requires_grad_(True)
    opt = torch.optim.Adam([raw], lr=lr)
    t0 = time.time()
    for step in range(steps + 1):
        opt.zero_grad()
        loss = nll_gjr(raw, r, x)
        loss.backward()
        opt.step()
        if step % every == 0:
            print(f'step {step:3d} | loss {loss.item():.5f} | params {[round(p.item(), 4) for p in unpack_gjr(raw)]} | {time.time() - t0:.0f}s')
    raw = raw.detach()
    return raw, nll_gjr(raw, r, x).item()

In [4]:
raw_gjr, loss_gjr = fit(torch.tensor([-1.0, 0.405, 0.916, 3.807], dtype=torch.float64), r_train, x_train)
print('GJR   omega, alpha, gamma, beta =', [round(p.item(), 4) for p in unpack_gjr(raw_gjr)[:4]], '| loss', round(loss_gjr, 5))
print('target (scipy):[0.0167, 0.0036, 0.1426, 0.9105] | loss 0.39807')

step   0 | loss 0.41042 | params [0.03, 0.0328, 0.1094, 0.8916, 0.0] | 2s
step  50 | loss 0.39837 | params [0.0192, 0.0106, 0.1421, 0.8998, 0.0] | 86s
step 100 | loss 0.39812 | params [0.0176, 0.0066, 0.1416, 0.9069, 0.0] | 170s
step 150 | loss 0.39809 | params [0.0169, 0.0054, 0.1418, 0.9089, 0.0] | 265s
step 200 | loss 0.39808 | params [0.0168, 0.0048, 0.142, 0.9097, 0.0] | 355s
step 250 | loss 0.39808 | params [0.0167, 0.0045, 0.1421, 0.91, 0.0] | 440s
step 300 | loss 0.39808 | params [0.0167, 0.0043, 0.1422, 0.9101, 0.0] | 532s
GJR   omega, alpha, gamma, beta = [0.0167, 0.0043, 0.1422, 0.9101] | loss 0.39808
target (scipy):                 [0.0167, 0.0036, 0.1426, 0.9105] | loss 0.39807


In [5]:
raw_gjrx, loss_gjrx = fit(torch.cat([raw_gjr, torch.zeros(1, dtype=torch.float64)]), r_train, x_train, steps=400)
print('GJR-X omega, alpha, gamma, beta, delta =', [round(p.item(), 4) for p in unpack_gjr(raw_gjrx)], '| loss', round(loss_gjrx, 5))
print('target (scipy):[0.0041, 0.0, 0.1846, 0.8007, 0.0661] | loss 0.38722')
print('alpha sits at its lower bound 0 in GJR-X, so torch will report a small positive alpha; the loss is what must match')

step   0 | loss 0.54411 | params [0.016, 0.0047, 0.1558, 0.9023, 0.0668] | 3s
step  50 | loss 0.39768 | params [0.0086, 0.0091, 0.3164, 0.8106, 0.0381] | 134s
step 100 | loss 0.39486 | params [0.0074, 0.0052, 0.2535, 0.8448, 0.0289] | 253s
step 150 | loss 0.39279 | params [0.0071, 0.0033, 0.2057, 0.868, 0.023] | 380s
step 200 | loss 0.39140 | params [0.0075, 0.0023, 0.1778, 0.8787, 0.0208] | 502s
step 250 | loss 0.39042 | params [0.0083, 0.0017, 0.1666, 0.8791, 0.0215] | 614s
step 300 | loss 0.38962 | params [0.0087, 0.0013, 0.1649, 0.8736, 0.0243] | 746s
step 350 | loss 0.38892 | params [0.0086, 0.0011, 0.1665, 0.8654, 0.0286] | 858s
step 400 | loss 0.38836 | params [0.0082, 0.0009, 0.1688, 0.8563, 0.0335] | 969s
GJR-X omega, alpha, gamma, beta, delta = [0.0082, 0.0009, 0.1688, 0.8563, 0.0335] | loss 0.38835
target (scipy):                        [0.0041, 0.0, 0.1846, 0.8007, 0.0661] | loss 0.38722
alpha sits at its lower bound 0 in GJR-X, so torch will report a small positive alpha; 

## 3. Frozen variance paths for every day

In [ ]:
@torch.no_grad()
def gjr_path(raw, r, x, h0):
    omega, alpha, gamma, beta, delta = unpack_gjr(raw)
    h = torch.empty_like(r)
    h[0] = h0
    for t in range(1, len(r)):
        neg = (r[t - 1] < 0).to(r.dtype)
        h[t] = omega + (alpha + gamma * neg) * r[t - 1] ** 2 + beta * h[t - 1] + delta * x[t - 1]
    return h


h_gjr = gjr_path(raw_gjr, r_all, x_all, r_train.var())
h_gjrx = gjr_path(raw_gjrx, r_all, x_all, r_train.var())
for name, h, ref in (('GJR', h_gjr, loss_gjr), ('GJR-X', h_gjrx, loss_gjrx)):
    chk = 0.5 * torch.mean(torch.log(h[1:N_TRAIN]) + r_all[1:N_TRAIN] ** 2 / h[1:N_TRAIN])
    print(f'{name}: loss from path on train days {chk.item():.5f} | fitted {ref:.5f}')

## 4. Features and windows
Base features: r, |r|, log h_GJR. VIX features: the same plus log VIX². Standardised with train-day statistics.
Target day t uses days t-22 … t-1. Windows are assigned to train / val / test by the target's date.

In [7]:
L = 22
feat_base = torch.stack([r_all, r_all.abs(), torch.log(h_gjr)], dim=1)          # (T, 3)
feat_vix = torch.cat([feat_base, torch.log(x_all).unsqueeze(1)], dim=1)          # (T, 4)


def standardise(Fr):
    mu, sd = Fr[:N_TRAIN].mean(dim=0), Fr[:N_TRAIN].std(dim=0)
    return ((Fr - mu) / sd).to(torch.float32)


Fb, Fv = standardise(feat_base), standardise(feat_vix)
targets = np.arange(L, len(r_all))
tdates = dates[targets]
split = np.where(tdates < TRAIN_END, 'train', np.where(tdates < VAL_END, 'val', 'test'))

Xb = torch.stack([Fb[t - L:t] for t in targets])        # (N, 22, 3)
Xv = torch.stack([Fv[t - L:t] for t in targets])        # (N, 22, 4)
y = r_all[targets].to(torch.float32)
hb = h_gjr[targets].to(torch.float32)                    # base variance for both networks
hx = h_gjrx[targets].to(torch.float32)                   # GJR-X, a baseline only

for i in (0, len(targets) // 2, len(targets) - 1):
    t = targets[i]
    assert torch.equal(Xv[i, -1], Fv[t - 1]) and torch.equal(Xv[i, 0], Fv[t - L])

masks = {s: torch.tensor(split == s) for s in ('train', 'val', 'test')}
print('Xb', tuple(Xb.shape), '| Xv', tuple(Xv.shape), '|', {s: int(mk.sum()) for s, mk in masks.items()},
      '(expected train 6279, val 1258, test 1674)')

Xb (9211, 22, 3) | Xv (9211, 22, 4) | {'train': 6279, 'val': 1258, 'test': 1674} (expected train 6279, val 1258, test 1674)


## 5. Model and training function
Same GINN as v1, now with a choice of hidden size and weight decay (L2 regularisation inside Adam).
Early stopping on validation. The starting point (epoch 0) is plain GJR, so a network that never beats GJR on validation returns GJR.

In [8]:
class GINN(nn.Module):
    def __init__(self, n_features, hidden):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x, h_base):
        out, _ = self.lstm(x)
        g = self.head(out[:, -1, :]).squeeze(-1)
        return h_base * torch.exp(g)


def qmle(h, r):
    return 0.5 * torch.mean(torch.log(h) + r ** 2 / h)


def train_ginn(X, base, hidden, wd, seed, max_epochs=3000, patience=300, lr=1e-3):
    tr, va = masks['train'], masks['val']
    Xtr, btr, ytr, Xva, bva, yva = X[tr], base[tr], y[tr], X[va], base[va], y[va]
    torch.manual_seed(seed)
    model = GINN(X.shape[2], hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_state, best_epoch = qmle(bva, yva).item(), copy.deepcopy(model.state_dict()), 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        loss = qmle(model(Xtr, btr), ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val = qmle(model(Xva, bva), yva).item()
            if val < best_val:
                best_val, best_state, best_epoch = val, copy.deepcopy(model.state_dict()), epoch
            if epoch - best_epoch >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, best_epoch

## 6. Small tuning grid, judged on validation only
hidden {8, 16} × weight decay {0, 1e-4, 1e-3}, for each network, seed 0.

In [10]:
print(f"validation loss  GJR {qmle(hb[masks['val']], y[masks['val']]).item():.5f} | GJR-X {qmle(hx[masks['val']], y[masks['val']]).item():.5f}")
grid = []
for name, X in (('GINN', Xb), ('GINN-VIX', Xv)):
    for hidden in (8, 16):
        for wd in (0.0, 1e-4, 1e-3):
            t0 = time.time()
            _, v, e = train_ginn(X, hb, hidden, wd, seed=0)
            grid.append({'model': name, 'hidden': hidden, 'weight_decay': wd, 'val_loss': round(v, 5),
                         'best_epoch': e, 'seconds': round(time.time() - t0)})
            print(grid[-1])
G = pd.DataFrame(grid)
best_cfg = G.loc[G.groupby('model')['val_loss'].idxmin()]
print('\nchosen settings:')
print(best_cfg.to_string(index=False))

validation loss  GJR 0.21869 | GJR-X 0.19414
{'model': 'GINN', 'hidden': 8, 'weight_decay': 0.0, 'val_loss': 0.20283, 'best_epoch': 210, 'seconds': 50}
{'model': 'GINN', 'hidden': 8, 'weight_decay': 0.0001, 'val_loss': 0.20275, 'best_epoch': 240, 'seconds': 49}
{'model': 'GINN', 'hidden': 8, 'weight_decay': 0.001, 'val_loss': 0.20241, 'best_epoch': 640, 'seconds': 78}
{'model': 'GINN', 'hidden': 16, 'weight_decay': 0.0, 'val_loss': 0.19741, 'best_epoch': 190, 'seconds': 42}
{'model': 'GINN', 'hidden': 16, 'weight_decay': 0.0001, 'val_loss': 0.20051, 'best_epoch': 300, 'seconds': 50}
{'model': 'GINN', 'hidden': 16, 'weight_decay': 0.001, 'val_loss': 0.20532, 'best_epoch': 430, 'seconds': 61}
{'model': 'GINN-VIX', 'hidden': 8, 'weight_decay': 0.0, 'val_loss': 0.17214, 'best_epoch': 200, 'seconds': 41}
{'model': 'GINN-VIX', 'hidden': 8, 'weight_decay': 0.0001, 'val_loss': 0.1721, 'best_epoch': 210, 'seconds': 43}
{'model': 'GINN-VIX', 'hidden': 8, 'weight_decay': 0.001, 'val_loss': 0.1706

## 7. Final models: chosen settings, 3 seeds, forecasts averaged
Averaging is done on g (the log correction), i.e. h = h_GJR · exp(mean g).

In [11]:
final = {}
for _, row in best_cfg.iterrows():
    X = Xb if row['model'] == 'GINN' else Xv
    gs = []
    for seed in (0, 1, 2):
        mdl, v, e = train_ginn(X, hb, int(row['hidden']), float(row['weight_decay']), seed)
        mdl.eval()
        with torch.no_grad():
            gs.append(torch.log(mdl(X, hb) / hb))
        print(f"{row['model']:8s} seed {seed} | val {v:.5f} | best epoch {e}")
    final[row['model']] = hb * torch.exp(torch.stack(gs).mean(dim=0))

GINN     seed 0 | val 0.19741 | best epoch 190
GINN     seed 1 | val 0.20334 | best epoch 180
GINN     seed 2 | val 0.20462 | best epoch 130
GINN-VIX seed 0 | val 0.16643 | best epoch 530
GINN-VIX seed 1 | val 0.16884 | best epoch 280
GINN-VIX seed 2 | val 0.16793 | best epoch 370


## 8. Results
Loss per split, Diebold-Mariano tests on the test days (Newey-West, 5 lags; negative = first model better), stress years.

In [12]:
preds = {'GJR': hb, 'GJR-X (VIX)': hx, 'GINN': final['GINN'], 'GINN-VIX': final['GINN-VIX']}


def day_loss(h):
    return 0.5 * (torch.log(h) + y ** 2 / h)


def dm_tstat(dd, lags=5):
    n, u = len(dd), dd - dd.mean()
    lrv = np.mean(u * u) + 2 * sum((1 - k / (lags + 1)) * np.mean(u[k:] * u[:-k]) for k in range(1, lags + 1))
    return dd.mean() / np.sqrt(lrv / n)


L_day = {k: day_loss(h).double().numpy() for k, h in preds.items()}
table = pd.DataFrame({s: {k: L_day[k][masks[s].numpy()].mean() for k in preds} for s in ('train', 'val', 'test')})
print('mean QMLE loss per day (lower is better)')
print(table.round(5).to_string())

te = masks['test'].numpy()
print('\nTest-period comparisons')
for a, b in (('GINN', 'GJR'), ('GINN-VIX', 'GJR-X (VIX)'), ('GJR-X (VIX)', 'GJR'), ('GINN-VIX', 'GINN')):
    dd = L_day[a][te] - L_day[b][te]
    print(f'  {a:12s} - {b:12s}: {dd.mean():+.5f} | DM t {dm_tstat(dd):+.2f}')

years = pd.DatetimeIndex(tdates).year.to_numpy()
print('\nStress years (test period), mean loss')
print(pd.DataFrame({yr: {k: L_day[k][te & (years == yr)].mean() for k in preds} for yr in (2020, 2022, 2025)}).round(5).to_string())

mean QMLE loss per day (lower is better)
               train      val     test
GJR          0.39750  0.21869  0.53215
GJR-X (VIX)  0.38773  0.19414  0.51751
GINN         0.37477  0.19933  0.52361
GINN-VIX     0.36896  0.16701  0.50540

Test-period comparisons
  GINN         - GJR         : -0.00853 | DM t -1.16
  GINN-VIX     - GJR-X (VIX) : -0.01211 | DM t -1.94
  GJR-X (VIX)  - GJR         : -0.01464 | DM t -2.03
  GINN-VIX     - GINN        : -0.01822 | DM t -1.92

Stress years (test period), mean loss
                2020     2022     2025
GJR          0.93848  0.97804  0.44184
GJR-X (VIX)  0.89137  0.98368  0.42560
GINN         0.95114  0.99552  0.40677
GINN-VIX     0.89545  0.99110  0.39515


In [13]:
os.makedirs('ginn_outputs', exist_ok=True)
out = pd.DataFrame({'date': tdates, 'split': split, 'r': y.numpy(), **{k: v.numpy() for k, v in preds.items()}})
out.to_csv('ginn_outputs/ginn_v2_daily.csv', index=False)
G.to_csv('ginn_outputs/ginn_v2_grid.csv', index=False)
print('saved ginn_outputs/ginn_v2_daily.csv and ginn_v2_grid.csv |', out.shape)

saved ginn_outputs/ginn_v2_daily.csv and ginn_v2_grid.csv | (9211, 7)
